# Section 6 — Advanced Concepts in Generative AI

Four production-grade techniques on top of the LLM basics from Section 5: Structured Outputs, Function Calling, Fine-Tuning, and Retrieval-Augmented Generation — applied to four concrete actuarial use cases.

Part of the EAA seminar *Machine Learning & Generative AI: A Hands-On Guide to Actuarial Practice* by Dr. Simon Hatzesberger (8–9 June 2026, Munich).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simonhatzesberger/ml-genai-actuarial-practice/blob/main/notebooks/06_genai_advanced_concepts/06_genai_advanced_concepts.ipynb)

Repository: [simonhatzesberger/ml-genai-actuarial-practice](https://github.com/simonhatzesberger/ml-genai-actuarial-practice) — Code under the [MIT License](https://github.com/simonhatzesberger/ml-genai-actuarial-practice/blob/main/LICENSE).

> **Provenance.** Sections 1–4 of this notebook are adapted from *GenAI Beyond the Basics* by the German Actuarial Association (Deutsche Aktuarvereinigung) — repository [DeutscheAktuarvereinigung/GenAI_Beyond_the_Basics](https://github.com/DeutscheAktuarvereinigung/GenAI_Beyond_the_Basics). The seminar version standardises on OpenAI's Responses API and adds a 2026 deprecation note on the OpenAI fine-tuning workflow.

This notebook builds on Section 5. Each of the four sections below tackles a single advanced technique on a single actuarial example:

| Section | Technique | Actuarial example |
|---|---|---|
| §1 | Structured Outputs | Extract policy fields from a RISCBAC auto-insurance contract |
| §2 | Function Calling | Let an LLM query an SQLite mortality table on demand |
| §3 | Fine-Tuning | Tailor a model to your domain — workflow + 2026 deprecation note |
| §4 | Retrieval-Augmented Generation | Extract KPIs (solvency ratio, discount rates, cyber-risk strategies) from an insurer's annual-report PDF |

## Contents

- [Learning objectives](#learning-objectives)
- [Setup](#setup)
- [1. Structured Outputs](#1-structured-outputs)
- [2. Function Calling](#2-function-calling)
- [3. Fine-Tuning](#3-fine-tuning)
- [4. Retrieval-Augmented Generation (RAG)](#4-retrieval-augmented-generation-rag)
- [Exercises](#exercises)
- [Summary](#summary)
- [Next steps](#next-steps)
- [References](#references)

## Learning objectives

By the end of this notebook you will be able to:

- Use **Structured Outputs** to constrain an LLM's response to a Pydantic-typed JSON schema, eliminating fragile post-hoc parsing. Apply it hands-on to policy-field extraction from a real (synthetic) car-insurance contract.
- Use **Function Calling** to expose deterministic Python code (database lookups, actuarial calculations) to an LLM, with the model deciding when to call. Apply it hands-on to a mortality-table query.
- Understand the **fine-tuning** workflow (data preparation → training → deployment), assess when it is the right tool, and know the modern open-weights alternative (PEFT / LoRA) given that OpenAI wound down its fine-tuning platform in May 2026.
- Build a small **RAG** pipeline (load → clean → chunk → embed → retrieve → augment → respond) and apply it to KPI extraction from a real insurer annual-report PDF.

## Setup

The cell below detects whether you are running on Google Colab. On Colab it installs the section's pinned dependencies; on a local install it assumes you have already done `pip install -r notebooks/06_genai_advanced_concepts/requirements.txt` inside your virtual environment.

For the cloud cells we use the OpenAI Responses API. Copy `.env.example` to `.env` in this folder and paste your `OPENAI_API_KEY`. If no key is set, every cloud cell skips gracefully — the dataset-loading and conceptual-code cells still run.

> **Note.** The RAG demo in §4 fetches the Generali Annual Report PDF (≈30 MB) from generali.com over the public internet at run time. If you are running offline, that cell will fail with a clear error — but the rest of the notebook is unaffected.

In [ ]:
# Detect environment
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# On Colab, install this section's pinned dependencies + download data files.
if IN_COLAB:
    !pip install -q -r https://raw.githubusercontent.com/simonhatzesberger/ml-genai-actuarial-practice/main/notebooks/06_genai_advanced_concepts/requirements.txt
    !mkdir -p data
    !wget -q -O data/mortality_tables.sqlite https://raw.githubusercontent.com/simonhatzesberger/ml-genai-actuarial-practice/main/notebooks/06_genai_advanced_concepts/data/mortality_tables.sqlite
    !wget -q -O data/riscbac_en_first10.jsonl https://raw.githubusercontent.com/simonhatzesberger/ml-genai-actuarial-practice/main/notebooks/06_genai_advanced_concepts/data/riscbac_en_first10.jsonl

# Imports
import os
import re
import json
import sqlite3
import base64
from io import BytesIO
from urllib.parse import urlparse, unquote
from typing import Any, List, Literal, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pydantic import BaseModel
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Local .env loading (skip silently if python-dotenv is not available).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
HAS_OPENAI_KEY = bool(OPENAI_API_KEY)

print(f"Setup complete. Environment: {'Colab' if IN_COLAB else 'local'}.")
print(f"OpenAI key detected: {HAS_OPENAI_KEY}.")
if not HAS_OPENAI_KEY:
    print("  -> Cloud cells will skip gracefully. Local/data cells still run.")

We reuse the same colour palette as the earlier section notebooks so figures stay visually consistent across the seminar.

In [ ]:
PRIMARY      = "#1F3A6E"      # Navy blue
ACCENT_RED   = "#C0504D"      # Rust
ACCENT_GREEN = "#4E7C59"      # Forest
GRAY_TEXT    = "#404040"      # Dark gray

sns.set_theme(
    style="whitegrid",
    palette=[PRIMARY, ACCENT_RED, ACCENT_GREEN, "#7A8DA8", "#9C7A7A"],
    rc={
        "axes.edgecolor":   GRAY_TEXT,
        "axes.labelcolor":  GRAY_TEXT,
        "xtick.color":      GRAY_TEXT,
        "ytick.color":      GRAY_TEXT,
        "axes.titlecolor":  PRIMARY,
        "axes.titleweight": "bold",
        "grid.color":       "#E5E5E5",
        "figure.facecolor": "white",
        "axes.facecolor":   "white",
    },
)

# Default cloud model (matches Section 5).
MODEL_DEFAULT     = "gpt-5.4-mini"
EMBEDDINGS_MODEL  = "text-embedding-3-small"

# OpenAI client (None if no key)
if HAS_OPENAI_KEY:
    from openai import OpenAI
    client = OpenAI()
else:
    client = None

## 1. Structured Outputs

LLMs are designed to produce human-readable prose, yet many actuarial workflows require **machine-readable data** that drops straight into code, spreadsheets, or databases. Structured Outputs solve this by constraining the model to return a response that conforms to a **predefined JSON schema** rather than free-form text. When the LLM reliably fills named fields with well-specified data types, the result can be parsed deterministically — removing the need for fragile regex parsers and ad-hoc clean-up scripts.

OpenAI's `text_format` parameter (the successor to the earlier `response_format`) is the recommended way to apply Structured Outputs. Combined with Pydantic, it enforces end-to-end adherence to your JSON schema. The same workflow applies to other providers — declare the schema, embed its definition in the prompt, and instruct the model to honour it.

### Why it matters for actuaries

- **Repeatability** — pricing, reserving, and reporting pipelines receive the same field names and formats every run.
- **Interoperability** — JSON can be consumed by spreadsheets, BI dashboards, or actuarial software (Prophet, ResQ, …) with no extra transformation layer.
- **Expressiveness** — nested objects and optional fields capture complex insurance concepts such as multi-vehicle policies, layered treaties, or cash-flow waterfalls.

### How it works

1. **Define the schema** — create a Pydantic model (or JSON schema) that represents the desired output.
2. **Send prompt + schema** — pass the model class via `text_format=` (Responses API).
3. **LLM validation** — the model fills in the fields and self-checks before responding.
4. **Parse and use** — `response.output_parsed` is a fully typed Python object.

### 1.1 Use case — Policy-field extraction from a RISCBAC auto contract

The [RISCBAC corpus](https://huggingface.co/datasets/davebulaval/RISCBAC) is an open dataset of 10,000 bilingual (FR/EN) synthetic auto-insurance contracts generated from the Québec standard form. The repo ships a 10-record English sample at `data/riscbac_en_first10.jsonl`.

A RISCBAC policy embeds structured facts that we want to lift into a JSON object: policy and endorsement numbers, the insured's personal details, vehicle information, coverage limits, premium amounts, and effective dates.

We tackle the task in three progressively stricter stages, *all on the same policy text*:

1. **Stage 1** — free-form extraction (no schema, naïve baseline).
2. **Stage 2** — flat Pydantic schema (five required fields).
3. **Stage 3** — nested Pydantic schema (mirrors real policy complexity).

In [ ]:
# Load the first English contract from the JSONL sample.
DATA_DIR = "data"
RISCBAC_PATH = os.path.join(DATA_DIR, "riscbac_en_first10.jsonl")

with open(RISCBAC_PATH, "r", encoding="utf-8") as f:
    first_line = f.readline()

policy_record = json.loads(first_line)
policy_text = policy_record["text"]

# Show a short excerpt so we know what we're working with.
print(policy_text[:600] + "\n\n... [truncated]")
print(f"\n(full document: {len(policy_text):,} characters)")

#### Stage 1 — Free-form extraction

We ask the model for the same five fields without any schema. Run the cell multiple times and notice the format drift: currency symbols may appear or disappear, separators change, sometimes the output is bullets and sometimes prose. Each response is human-readable but unparseable.

In [ ]:
instructions_naive = """
You will be given a free-form description of a car insurance contract.
Your job is to extract the driver's last name, the driver's first name, the vehicle, the premium, and the contract period.
"""

if HAS_OPENAI_KEY:
    r = client.responses.create(
        model=MODEL_DEFAULT,
        input=policy_text,
        instructions=instructions_naive,
    )
    print(r.output_text)
else:
    print("[skipped — no OPENAI_API_KEY] Example expected output:")
    print("""Driver last name: Tremblay
Driver first name: Marie
Vehicle: 2018 Toyota Corolla LE
Premium: $1,234
Contract period: 12 months""")

#### Stage 2 — Flat Pydantic schema

We define a simple Pydantic model `CarInsuranceContract_basic` with five required fields and pass it via `text_format=`. The model is now constrained to return strictly valid JSON that matches the schema — no extra commentary, no missing fields, no format drift.

In [ ]:
class CarInsuranceContract_basic(BaseModel):
    driver_last_name: str
    driver_first_name: str
    vehicle_make_and_model: str
    premium: int
    contract_period: str


instructions_basic = """
Extract the following fields from the policy text and return a strictly valid JSON object
matching the `CarInsuranceContract_basic` schema:

- driver_last_name: string
- driver_first_name: string
- vehicle_make_and_model: string
- premium: integer
- contract_period: string

Do not add any commentary or extra keys.
"""

if HAS_OPENAI_KEY:
    r = client.responses.parse(
        model=MODEL_DEFAULT,
        input=policy_text,
        instructions=instructions_basic,
        text_format=CarInsuranceContract_basic,
    )
    parsed = r.output_parsed
    print("Parsed Python object:", parsed)
    print()
    print("As JSON:")
    print(json.dumps(parsed.model_dump(), indent=2))
else:
    print("[skipped — no OPENAI_API_KEY]")

#### Stage 3 — Nested Pydantic schema

A richer hierarchy mirrors real policy complexity. We add nested sub-models, optional fields, an enum-constrained month, and a list-typed field.

In [ ]:
Month = Literal[
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

class BirthDate(BaseModel):
    year: int
    month: Month
    day: int

class Driver(BaseModel):
    last_name: str
    first_name: str
    second_first_name: Optional[str] = None
    birth_date: BirthDate

class Vehicle(BaseModel):
    make_and_model: str
    serial_number: str

class Miscellaneous(BaseModel):
    premium: int
    tax: float
    currency: str
    damage_to_insured_vehicles: List[str]

class CarInsuranceContract_advanced(BaseModel):
    driver: Driver
    vehicle: Vehicle
    miscellaneous: Miscellaneous


instructions_advanced = """
Extract the relevant fields from the policy text and return a strictly valid JSON object
matching the `CarInsuranceContract_advanced` Pydantic schema. Do not add commentary.

If a sub-field is not present in the text, return null for optional fields (e.g. second_first_name)
or an empty list for list-typed fields.
"""

if HAS_OPENAI_KEY:
    r = client.responses.parse(
        model=MODEL_DEFAULT,
        input=policy_text,
        instructions=instructions_advanced,
        text_format=CarInsuranceContract_advanced,
    )
    parsed = r.output_parsed
    print(json.dumps(parsed.model_dump(), indent=2))
else:
    print("[skipped — no OPENAI_API_KEY]")

### 1.2 Further actuarial applications

- **Multiclass classification** — predefined classes for document type, damage category, complaint topic, or claim severity. Always returns one of the labels you specified.
- **Fraud detection and pattern recognition** — turn narrative claims notes into structured records, ready to feed a downstream anomaly model.
- **Claims settlement automation** — extract structured fields from adjuster notes / customer communications to drive payment, reserving, and audit workflows.

For background see OpenAI's [Structured Outputs guide](https://platform.openai.com/docs/guides/structured-outputs) and the [OpenAI Cookbook structured-outputs intro](https://cookbook.openai.com/examples/structured_outputs_intro).

## 2. Function Calling

LLMs excel at understanding and generating natural-language text, but they cannot execute deterministic business logic — database look-ups, actuarial calculations, API calls — on their own. **Function Calling** (sometimes *Tool Calling*) solves this: you expose a controlled set of back-end functions to the model, and the model decides at generation time whether to:

- answer the user directly, or
- request that the application run one of the provided functions and feed the result back into the conversation.

This pattern combines the LLM's intent-recognition with the reliability, speed, and auditability of classical code.

### Why it matters for actuaries

- **Repeatability & compliance** — deterministic functions ensure that premium calculations, reserving routines, or regulatory checks are performed exactly the same way every time.
- **Traceability** — every function invocation, its arguments, and its result are logged — an audit trail.
- **Scalability** — wrap anything from a simple NPV formula to a high-performance risk engine as a callable tool.
- **Lower prompting effort** — let code do the heavy lifting; the model just decides *what* to call and *explains why*.

### How it works — the six-step round trip

1. **Application → LLM**: send the user prompt + JSON definitions of available functions.
2. **LLM reasoning**: decide whether to respond directly or invoke a tool.
3. **LLM → Application**: return either a normal assistant message or a `function_call` with `name` + `arguments`.
4. **Application layer**: parse the message and execute the requested function with the supplied arguments.
5. **Application → LLM**: re-call the LLM, passing the original prompt + the function's result.
6. **LLM reasoning + final response**: integrate the result into the answer, or trigger another tool call.

### 2.1 Use case — Mortality-table query via SQLite

The repo ships a small SQLite database (`data/mortality_tables.sqlite`) with one table named `mortality_table` holding columns `(mortality_table, gender, age, probability)`. We expose a Python function `fetch_probabilities(mortality_table, start_age, end_age, sex)` and let the LLM decide when to call it. The user prompt asks for an analysis of male mortality between ages 85 and 100 from the German private-health table *PKV-Sterbetafel 2025*; the model recognises it needs the data, calls the function, then writes a markdown analysis.

In [ ]:
DB_PATH = os.path.join(DATA_DIR, "mortality_tables.sqlite")

def fetch_probabilities(mortality_table: str, start_age: int, end_age: int, sex: str = "FEMALE"):
    """Return mortality probabilities from the SQLite DB for [start_age..end_age]."""
    sex = sex.upper()
    safe_table = mortality_table.replace("'", "''")  # basic escape
    sql = """
        SELECT age, probability
          FROM mortality_table
         WHERE mortality_table = ?
           AND gender = ?
           AND age BETWEEN ? AND ?
         ORDER BY age
    """
    conn = sqlite3.connect(DB_PATH)
    try:
        cur = conn.cursor()
        cur.execute(sql, (safe_table, sex, start_age, end_age))
        rows = cur.fetchall()
    finally:
        conn.close()
    return [{"age": age, "probability": prob} for age, prob in rows]


# JSON tool definition the LLM sees.
tools = [
    {
        "type": "function",
        "name": "fetch_probabilities",
        "description": "Fetch mortality probabilities from an SQLite database for an age range and gender.",
        "parameters": {
            "type": "object",
            "properties": {
                "mortality_table": {
                    "type": "string",
                    "description": "Name of the mortality table (e.g. 'PKV-Sterbetafel 2025').",
                },
                "start_age": {"type": "integer", "description": "Start age (inclusive)."},
                "end_age":   {"type": "integer", "description": "End age (inclusive)."},
                "sex": {
                    "type": "string",
                    "enum": ["MALE", "FEMALE"],
                    "description": "Gender to filter by (defaults to FEMALE).",
                    "default": "FEMALE",
                },
            },
            "required": ["mortality_table", "start_age", "end_age"],
        },
    }
]

# Quick local smoke test of the function (no API needed).
sample = fetch_probabilities("PKV-Sterbetafel 2025", start_age=85, end_age=90, sex="MALE")
print("Local function call returns:")
for row in sample:
    print(f"  age={row['age']:3d}  q={row['probability']:.6f}")

#### Steps 1–2 — Send prompt + tools; let the model decide

The user prompt explicitly asks for mortality probabilities in a markdown table plus a brief trend analysis. The model has the choice to either call `fetch_probabilities` or to answer from its (limited, training-cutoff) knowledge.

In [ ]:
user_prompt = (
    "Please fetch the probabilities for men between age 85 and one hundred from the "
    "mortality table 'PKV-Sterbetafel 2025'. First, present the data in a markdown-formatted table. "
    "Then write two or three sentences on the trend (smoothness, monotonicity, magnitude)."
)

input_messages = [{"role": "user", "content": user_prompt}]

if HAS_OPENAI_KEY:
    response = client.responses.create(
        model=MODEL_DEFAULT,
        input=input_messages,
        tools=tools,
    )
    first_chunk = response.output[0]
    if hasattr(first_chunk, "name"):
        print(f"Model decided to CALL function: {first_chunk.name}")
    else:
        print("Model returned a direct message (no function call).")
else:
    print("[skipped — no OPENAI_API_KEY] Expected: Model decided to CALL function: fetch_probabilities")

#### Steps 3–4 — Extract the call and execute it locally

In [ ]:
if HAS_OPENAI_KEY:
    tool_call = response.output[0]                # has .name, .call_id, .arguments
    args = json.loads(tool_call.arguments)
    print("Model wants to call:", tool_call.name)
    print("With arguments:", args)

    fn_result = fetch_probabilities(**args)
    print(f"\nFunction returned {len(fn_result)} rows. First few:")
    for row in fn_result[:5]:
        print(f"  age={row['age']:3d}  q={row['probability']:.6f}")
else:
    print("[skipped — no OPENAI_API_KEY]")

#### Steps 5–6 — Re-invoke the LLM with the result; produce the final analysis

In [ ]:
if HAS_OPENAI_KEY:
    # Provide the function output back to the LLM as context for the second call.
    input_messages.append(tool_call)
    input_messages.append({
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(fn_result),
    })

    response_2 = client.responses.create(
        model=MODEL_DEFAULT,
        input=input_messages,
        tools=tools,
    )
    print(response_2.output_text)
else:
    print("[skipped — no OPENAI_API_KEY]")

### 2.2 Further actuarial applications

- **Dynamic tariff calculation** — business users change coverages / deductibles in chat; the model calls a tariff function for instant requotes.
- **Customer service with live data** — renewal dates, payment history, coverage details — answered from the policy DB, not from training data.
- **Risk scenario simulation** — risk manager describes a stress test in natural language; the model calls a Monte Carlo engine.
- **Automated policy issuance** — underwriter describes the case; the model calls a policy-generation function and returns a draft.

For background see OpenAI's [Function Calling guide](https://platform.openai.com/docs/guides/function-calling) and the [Cookbook example](https://cookbook.openai.com/examples/how_to_call_functions_with_chat_models).

> **Connection to Structured Outputs.** Tool definitions are themselves JSON schemas. Once you are comfortable with §1, the `parameters` block of a tool definition is just a Pydantic-like schema you have already seen.

## 3. Fine-Tuning

Fine-tuning adapts a pre-trained LLM to your specific domain by continuing training on a curated set of labelled examples. Unlike prompt engineering — where you provide context and examples at inference time — fine-tuning adjusts the model's internal weights, yielding permanent improvements that no longer require repeated context.

> **2026 deprecation notice.** On **2026-05-07**, OpenAI [announced](https://community.openai.com/t/openai-is-winding-down-the-fine-tuning-api-and-platform-discussion-thread/1380522) the wind-down of its self-serve fine-tuning platform. New users can no longer onboard; existing users can still create supervised fine-tuning (SFT) and DPO jobs on `gpt-4.1`, `gpt-4.1-mini`, and reinforcement fine-tuning (RFT) on `o4-mini`. **GPT-5.x models are not available for fine-tuning.** Inference on existing fine-tuned models will continue until the base model is itself deprecated.
>
> In practice this means: for new actuarial fine-tuning work in mid-2026 and beyond, the path of least resistance is **open-weights models + LoRA / PEFT** (see §3.3 below). The OpenAI workflow remains pedagogically useful as a reference, and the JSONL data format generalises directly to many other vendors.

### Why it (still) matters for actuaries

- **Regulatory precision** — reproduce mandatory wording (Solvency II disclosures, IFRS 17 notes) verbatim; minimise hallucinations on jargon.
- **Data privacy / efficiency** — proprietary inputs (loss triangles, tariff tables) are used only during training; inference no longer needs to repeatedly expose sensitive data in the prompt.
- **Consistent decision-making** — underwriting or claims triage follow the same trained logic every time.
- **Lower cost / latency** — a fine-tuned cheap-tier model often matches frontier accuracy on a narrow task, at a fraction of the cost.

### 3.1 The three-step workflow

**Step 1 — Data preparation.** Collect representative prompt → response pairs that reflect your task, format them as JSONL. OpenAI recommends ≥50 high-quality examples per task; the absolute minimum is 10. Stratify across categories and hold out a validation split.

**Step 2 — Training.** Upload the JSONL files and launch a job. Configure hyperparameters (learning rate, batch size, epochs) if needed.

**Step 3 — Deployment & governance.** The job emits a model ID (e.g. `ft:gpt-4.1-mini:org::abc123`). Call inference with that ID via the standard Responses API.

### 3.2 Step 1 — Data prep (small worked example)

To keep this section reproducible without Kaggle or vision setup, we use a tiny **text** classification example: routing a customer complaint to one of four queues. The same JSONL format works for any text-in / text-out task.

In [ ]:
# Tiny synthetic training set. In practice you'd ship hundreds of these.
training_examples = [
    ("My agent never returns my calls.",                                  "SERVICE"),
    ("Forty-minute hold to reach a human.",                               "SERVICE"),
    ("Renewal premium up 35% — no explanation.",                          "PRICING"),
    ("You doubled my premium with no warning.",                           "PRICING"),
    ("Six weeks and no decision on my motor claim.",                      "CLAIMS-HANDLING"),
    ("Adjuster lowballed my contents claim.",                             "CLAIMS-HANDLING"),
    ("Can I add my partner as a named driver?",                           "OTHER"),
    ("What does 'aggregate excess' mean in my policy?",                   "OTHER"),
]

SYSTEM_PROMPT = (
    "You are a complaint router. Classify the input into exactly one of: "
    "SERVICE, CLAIMS-HANDLING, PRICING, OTHER. Return only the label."
)

# OpenAI fine-tuning expects one JSON object per line, each with a `messages` list.
def to_jsonl_record(user_text: str, assistant_label: str) -> dict:
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_text},
            {"role": "assistant", "content": assistant_label},
        ]
    }

train_jsonl_path = "data/complaint_router_train.jsonl"
with open(train_jsonl_path, "w", encoding="utf-8") as f:
    for text, label in training_examples:
        f.write(json.dumps(to_jsonl_record(text, label)) + "\n")

print(f"Wrote {len(training_examples)} examples to {train_jsonl_path}.")
print("\nFirst record (pretty-printed):")
with open(train_jsonl_path, "r", encoding="utf-8") as f:
    print(json.dumps(json.loads(f.readline()), indent=2))

### 3.3 Step 2 — Training (reference code — DO NOT RUN)

The cell below shows the API call shape for a supervised fine-tuning job on OpenAI. The `RUN_FINE_TUNING_JOB` flag is gated `False` by default — running this in mid-2026 will fail for new users due to the platform wind-down, and burns credit for existing users. The cell is included as documentation only.

In [ ]:
RUN_FINE_TUNING_JOB = False  # Leave False unless you know what you're doing.

if RUN_FINE_TUNING_JOB and HAS_OPENAI_KEY:
    # Upload training file (and validation file, if you have one).
    train_file = client.files.create(
        file=open(train_jsonl_path, "rb"),
        purpose="fine-tune",
    )

    # Launch the job. On the legacy chat-completions surface; SFT only.
    # Available base models in mid-2026 (existing users only): "gpt-4.1", "gpt-4.1-mini".
    job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        model="gpt-4.1-mini",
    )
    print("Submitted fine-tuning job:", job.id)
    print("Track progress via client.fine_tuning.jobs.retrieve(job.id).")
else:
    print("[skipped]  RUN_FINE_TUNING_JOB is False — this is reference code only.")
    print("           (As of 2026-05-07, OpenAI no longer accepts new fine-tuning users.)")

### 3.4 The modern alternative — Open-weights + LoRA / PEFT

For new fine-tuning work in 2026, the path of least resistance is parameter-efficient fine-tuning on **open-weights** models:

- **Base model.** A capable open-weights model from Hugging Face — Llama 4 Scout, Mistral Small 4, Qwen 3.5, Gemma 4, etc. The model card lists license terms; check before commercial use.
- **Adapter method.** **LoRA** (Low-Rank Adaptation) or **QLoRA** (4-bit quantised LoRA) — training adapter matrices of a few million parameters on top of a frozen base model. Captures ~95% of full-fine-tuning quality at ~1% of the compute and memory.
- **Library.** Hugging Face [PEFT](https://huggingface.co/docs/peft) sits on top of `transformers` + `accelerate`. The training loop is ten lines of code; the rest is data prep.
- **Serving.** Once trained, the adapter is a few-megabyte file. Serve via vLLM (production) or `transformers` (prototyping). The adapter loads on top of the base weights at startup, no merging required.
- **Data format.** Same conceptual JSONL — system / user / assistant turns — but PEFT can also consume plain `(prompt, completion)` pairs or `prompt → response` records depending on the training script.

Pseudo-code for a LoRA fine-tune on Llama-4-Scout-Instruct (illustrative; do not run here):

```python
# pip install transformers peft accelerate datasets bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-4-Scout-17B-16E-Instruct")
model     = AutoModelForCausalLM.from_pretrained(..., load_in_4bit=True)

config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, config)

ds = load_dataset("json", data_files="data/complaint_router_train.jsonl")
# ... standard HF Trainer loop ...
```

For full walk-throughs see the [PEFT documentation](https://huggingface.co/docs/peft) and the [Hugging Face fine-tuning tutorial](https://huggingface.co/learn/cookbook/en/fine_tuning_code_llm_on_single_gpu).

### 3.5 Historical case study — Car-damage classification (vision SFT)

The source notebook (Deutsche Aktuarvereinigung, *GenAI Beyond the Basics*) demonstrates vision SFT on a Kaggle car-damage dataset: ~1,500 images across six classes (crack, scratch, tire flat, dent, glass shatter, lamp broken), 60 / 20 / 20 split, fine-tune a vision-enabled GPT-4o, compare accuracy against the off-the-shelf baseline. That workflow is now historical — the OpenAI vision-SFT endpoint is closed to new users — but it remains a useful template for the equivalent task on open-weights vision models (LLaVA 1.6, Qwen2-VL, …) via PEFT.

### 3.6 Further actuarial applications

- **Tone-of-voice customisation** — claim emails, broker communications, regulator-facing notes — fine-tune for the firm's voice.
- **Domain terminology** — bake in Solvency-II / IFRS-17 / firm-internal terminology so the model uses it consistently without prompt repetition.
- **Consistent decision-making** — underwriting referrals, claims-triage decisions, fraud flags — fine-tune a small open-weights model on a labeled history.
- **Cost compression** — a LoRA-fine-tuned 7B open-weights model can match a frontier-cheap-tier model on narrow tasks at fraction of the inference cost.

## 4. Retrieval-Augmented Generation (RAG)

RAG combines LLMs with external data sources to produce responses that are not only linguistically coherent but **factually grounded**. Instead of relying entirely on the knowledge embedded in the model's weights, the system selectively accesses external resources — databases, document repositories, APIs — whenever specific or up-to-date information is required.

By integrating the natural-language strengths of LLMs with the precision of external data, RAG outputs are:

- **Current** — external information is updated continuously, no model retraining required.
- **Accurate** — relevant authoritative chunks of source text drive the answer.
- **Flexible** — add or swap sources at any time.

### The five stages

1. **Indexing** — split source materials (reports, policies, regulations) into chunks; turn each chunk into a semantic embedding; store in a vector index.
2. **Vectorize & search** — embed the user query the same way; find the most semantically similar chunks via cosine similarity (or other metrics).
3. **Retrieve** — pull the top-N chunks above a similarity threshold.
4. **Augment** — concatenate retrieved chunks into the LLM prompt as grounding context.
5. **Generate** — call the LLM with the augmented prompt — typically with Structured Outputs (§1) to get machine-readable answers back.

> **Variants.** Microsoft's **GraphRAG** adds a knowledge graph over the chunks. **AgenticRAG** lets an LLM-agent iteratively refine the retrieval query before generating the final answer. Both are out of scope here.

### 4.1 Use case — KPI extraction from an insurer annual-report PDF

We extract three KPIs from the *Generali Group Annual Integrated Report 2024* (≈30 MB PDF served from generali.com):

1. **Solvency capital ratio** for 2024, plus regulatory framework (Solvency II or SST).
2. **Discount rates** by duration in EUR.
3. **Cyber-risk mitigation strategies** as a list.

The same pipeline scales horizontally to any number of insurer reports for cross-company comparison.

In [ ]:
# RAG pipeline constants — tune these for your corpus and budget.
CHUNK_SIZE = 2000  # characters per chunk
OVERLAP    = 300   # characters of overlap between adjacent chunks
TOP_N      = 5     # number of chunks to retrieve per query
THRESHOLD  = 0.5   # cosine-similarity cutoff

GENERALI_REPORT_URL = (
    "https://www.generali.com/doc/jcr:259c5d6e-46f7-4a43-9512-58e5dcbd2a56/"
    "Annual%20Integrated%20Report%20and%20Consolidated%20Financial%20Statements%202024_"
    "Generali%20Group_final_interactive.pdf/lang:en/"
    "Annual_Integrated_Report_and_Consolidated_Financial_Statements_2024_Generali_Group_final_interactive.pdf"
)

print(f"Chunk size: {CHUNK_SIZE} chars, overlap: {OVERLAP}, top-N: {TOP_N}, sim cutoff: {THRESHOLD}.")

#### Stage 1 — Preprocessing (load → clean → chunk → embed)

In [ ]:
import requests
import fitz  # PyMuPDF — installed as `pymupdf`, imported as `fitz`

def load_pdf_from_url(url: str) -> dict:
    """Fetch a PDF from a URL into memory and extract its text. Returns {filename, text}."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    filename = unquote(urlparse(url).path.split("/")[-1])
    doc = fitz.open(stream=resp.content, filetype="pdf")
    full_text = "\n".join(page.get_text() for page in doc)
    return {"filename": filename, "text": full_text}


def clean_text(text: str) -> str:
    """Strip extra whitespace and empty lines."""
    text = re.sub(r"[ \t]+", " ", text)
    text = text.strip()
    text = re.sub(r"\n\s*\n", "\n", text)
    return text


def create_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> List[str]:
    """Split text into overlapping fixed-character chunks."""
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap.")
    step = chunk_size - overlap
    return [text[i: i + chunk_size] for i in range(0, len(text), step)]


def embed_text(chunk: str, model: str = EMBEDDINGS_MODEL) -> Optional[List[float]]:
    """Generate a single embedding vector for one chunk."""
    if not HAS_OPENAI_KEY:
        return None
    response = client.embeddings.create(model=model, input=chunk, encoding_format="float")
    return response.data[0].embedding


def build_document_index(text: str) -> pd.DataFrame:
    """Clean + chunk + embed a single document. Returns DataFrame with content + embedding."""
    cleaned = clean_text(text)
    chunks  = create_chunks(cleaned)
    rows = []
    for chunk in tqdm(chunks, desc="embedding chunks"):
        emb = embed_text(chunk)
        if emb is not None:
            rows.append({"content": chunk, "embedding": emb})
    return pd.DataFrame(rows)

print("Helper functions defined.")

In [ ]:
# Load the report PDF and build the embedding index.
# This cell takes a couple of minutes on first run (PDF is ~30 MB, hundreds of chunks).

if HAS_OPENAI_KEY:
    try:
        report = load_pdf_from_url(GENERALI_REPORT_URL)
        print(f"Loaded {report['filename']} — {len(report['text']):,} chars.")
        company_embeddings = {"Generali": build_document_index(report["text"])}
        print(f"\nIndex built: {len(company_embeddings['Generali'])} chunks for Generali.")
    except Exception as e:
        print(f"PDF download or embedding failed: {e}")
        company_embeddings = {}
else:
    print("[skipped — no OPENAI_API_KEY]  RAG pipeline needs embeddings; cannot proceed without a key.")
    company_embeddings = {}

#### Stage 2 — Prompt augmenting (define queries, retrieve top-N)

In [ ]:
# A persistent system prompt that frames the structured-extraction task.
system_prompt_rag = """
You are an AI assistant specialised in extracting and structuring key financial and risk
insights from annual reports of European insurance companies. Use retrieval-augmented
generation to ground your answers in the provided text snippets; apply structured
outputs to return data in the exact schema specified. If data is missing or ambiguous,
return 'NA' for the affected field rather than guessing.
""".strip()

# Three targeted queries — one per KPI.
query_solvency = (
    "Extract the group's solvency capital ratio in percentage for 2024, along with the "
    "regulatory framework (Solvency II or SST)."
)
query_discount = (
    "Extract the discount rates for insurance contract liabilities in 2024, currency EUR. "
    "For each duration (1, 5, 10, 20, 40 years if available), extract the rate in percentage. "
    "Use rates as of 2024-12-31. Assume non-VFA / unit-linked / liquid products if not specified."
)
query_cyber = (
    "Extract the insurer's documented approach to cyber-risk assessment and mitigation. "
    "Return each policy, process, or control as a separate text item."
)

queries = [
    ("solvency", query_solvency),
    ("discount", query_discount),
    ("cyber",    query_cyber),
]
print("Queries defined.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_top_chunks(
    index_df: pd.DataFrame,
    query: str,
    top_n: int = TOP_N,
    similarity_cutoff: float = THRESHOLD,
) -> str:
    """Return the top-N most similar chunks from the document index, concatenated."""
    if not HAS_OPENAI_KEY or index_df.empty:
        return "No relevant content found."

    query_emb = embed_text(query)
    if query_emb is None:
        return "No relevant content found."

    qv = np.array(query_emb).reshape(1, -1)
    sims = index_df["embedding"].apply(lambda v: cosine_similarity(qv, np.array(v).reshape(1, -1))[0][0])
    filtered = index_df.assign(similarity=sims)
    filtered = filtered[filtered["similarity"] > similarity_cutoff]
    if filtered.empty:
        return "No relevant content found."

    top = filtered.nlargest(top_n, "similarity")
    return "\n\n---\n\n".join(top["content"].tolist())


# Retrieve chunks for each query.
retrieved = {}
if company_embeddings.get("Generali") is not None and not company_embeddings["Generali"].empty:
    for key, q in queries:
        retrieved[key] = retrieve_top_chunks(company_embeddings["Generali"], q)
    print("Retrieved chunks for all three queries.")
    print(f"\nExample — solvency-query chunks (truncated):\n{retrieved['solvency'][:600]}...")
else:
    print("[skipped — no embeddings index]")

#### Stage 3 — Response generation (augmented prompt + Structured Outputs)

For each KPI we define a Pydantic schema, build an augmented prompt (system prompt + retrieved chunks + query + schema), and call the Responses API with `text_format=` to get back a strongly-typed Python object.

In [ ]:
# One schema per KPI.

class SolvencyRatio(BaseModel):
    capital_ratio: int                                   # solvency ratio in %
    regulatory_framework: Literal["Solvency II", "SST"]

class DiscountRatePerDuration(BaseModel):
    duration_in_years: int
    discount_rate: float                                 # rate in percentage points (e.g. 2.47)

class DiscountRates(BaseModel):
    currency: str
    discount_rates_per_duration: List[DiscountRatePerDuration]

class CyberRiskStrategies(BaseModel):
    strategies: List[str]


def query_with_rag(query_key: str, query_text: str, schema: type) -> Any:
    """Compose an augmented prompt + call the LLM with text_format=schema. Returns the parsed object."""
    if not HAS_OPENAI_KEY:
        return None
    chunks = retrieved.get(query_key, "No relevant content found.")
    augmented = (
        f"Query:\n{query_text}\n\n"
        f"=========================================================================\n"
        f"Data snippets:\n{chunks}\n\n"
        f"=========================================================================\n"
        f"Extract the relevant information from the snippets and return a strictly "
        f"valid JSON object matching the schema. Use 'NA' (as a string) if a field "
        f"cannot be determined from the snippets."
    )
    r = client.responses.parse(
        model=MODEL_DEFAULT,
        input=augmented,
        instructions=system_prompt_rag,
        text_format=schema,
        temperature=0.0,
    )
    return r.output_parsed


# Run all three queries.
if HAS_OPENAI_KEY and retrieved:
    solvency_result  = query_with_rag("solvency", query_solvency, SolvencyRatio)
    discount_result  = query_with_rag("discount", query_discount, DiscountRates)
    cyber_result     = query_with_rag("cyber",    query_cyber,    CyberRiskStrategies)

    print("=== Solvency capital ratio ===")
    print(json.dumps(solvency_result.model_dump(), indent=2))
    print("\n=== Discount rates ===")
    print(json.dumps(discount_result.model_dump(), indent=2))
    print("\n=== Cyber-risk strategies ===")
    print(json.dumps(cyber_result.model_dump(), indent=2))
else:
    print("[skipped — no OPENAI_API_KEY]")

### 4.2 Further actuarial applications

- **Cross-company KPI scraping** — run the same pipeline across multiple insurer annual reports for peer benchmarking.
- **Internal-document Q&A** — point RAG at your firm's policy wordings, underwriting manuals, claims-handling SOPs. Combine with §1 Structured Outputs for machine-readable answers.
- **Regulatory text retrieval** — Solvency II Delegated Acts, IFRS 17 standards, GDPR guidelines — embed once, ask many questions.
- **Reserving-committee assistant** — ground discussion questions in the actual run-off triangles, model-validation reports, and prior-year actuarial-opinion text.

For background see the [LangChain RAG overview](https://python.langchain.com/docs/tutorials/rag/), the [LlamaIndex docs](https://docs.llamaindex.ai/), and Microsoft Research's [GraphRAG paper](https://arxiv.org/abs/2404.16130).

## Exercises

Three short exercises — one for the three techniques most participants will reach for in production (Structured Outputs, Function Calling, RAG). Solutions are inlined as code cells beneath each exercise.

### Exercise 1 — Extend the Structured Output schema

Take `CarInsuranceContract_advanced` from §1 Stage 3 and add a new sub-model `Coverage` with fields:

- `coverage_type: str`   (e.g. *"Civil liability"*, *"Collision"*)
- `limit: int`           (in the policy's currency)
- `deductible: int`      (in the policy's currency)

Attach a `List[Coverage]` field named `coverages` to the top-level `CarInsuranceContract_advanced`. Re-run the extraction on the same policy text and print the result.

In [ ]:
# Solution

class Coverage(BaseModel):
    coverage_type: str
    limit: int
    deductible: int

class CarInsuranceContract_extended(BaseModel):
    driver: Driver
    vehicle: Vehicle
    miscellaneous: Miscellaneous
    coverages: List[Coverage]


instructions_extended = """
Extract all relevant fields from the policy text and return a strictly valid JSON object
matching the `CarInsuranceContract_extended` schema. For `coverages`, include every
coverage section you can identify with its limit (currency-amount) and deductible
(currency-amount). Use 0 for deductible if none is stated. Do not add commentary.
"""

if HAS_OPENAI_KEY:
    r = client.responses.parse(
        model=MODEL_DEFAULT,
        input=policy_text,
        instructions=instructions_extended,
        text_format=CarInsuranceContract_extended,
    )
    print(json.dumps(r.output_parsed.model_dump(), indent=2))
else:
    print("[skipped — no OPENAI_API_KEY]")

### Exercise 2 — Add a second tool

The §2 mortality demo exposed only one function. Add a second one — `compute_life_expectancy(mortality_table, start_age, sex)` — that returns the actuarial life expectancy at `start_age` from the mortality table. (Approximation: sum of survival probabilities from `start_age` to age 100, plus a half-year-of-life correction at the end.)

Expose both tools to the LLM and ask: *"For a 65-year-old man on PKV-Sterbetafel 2025, give me the table of mortality probabilities from age 85 to 100 AND his current life expectancy."* The model should call **both** functions and integrate the results.

In [ ]:
# Solution

def compute_life_expectancy(mortality_table: str, start_age: int, sex: str = "FEMALE") -> float:
    """Curtate life expectancy + 0.5 correction. Simple sum-of-survival-probabilities approach."""
    rows = fetch_probabilities(mortality_table, start_age=start_age, end_age=120, sex=sex)
    p_survive = 1.0
    expected_years = 0.0
    for r in rows:
        q = r["probability"]
        p_survive *= (1.0 - q)
        expected_years += p_survive
    return round(expected_years + 0.5, 2)


tools_extended = tools + [
    {
        "type": "function",
        "name": "compute_life_expectancy",
        "description": "Compute approximate curtate life expectancy at a given start age from a mortality table.",
        "parameters": {
            "type": "object",
            "properties": {
                "mortality_table": {"type": "string", "description": "Name of the mortality table."},
                "start_age":       {"type": "integer", "description": "Age at which to compute the life expectancy."},
                "sex":             {"type": "string", "enum": ["MALE", "FEMALE"], "default": "FEMALE"},
            },
            "required": ["mortality_table", "start_age"],
        },
    }
]

# Local sanity check
sample_le = compute_life_expectancy("PKV-Sterbetafel 2025", start_age=65, sex="MALE")
print(f"Local life expectancy at 65 (male, PKV 2025) ≈ {sample_le} years")

# Build a multi-tool prompt and let the LLM call both.
user_prompt_ex2 = (
    "For a 65-year-old man on PKV-Sterbetafel 2025, give me the mortality probabilities "
    "from age 85 to 100 in a markdown table, AND his current life expectancy at 65. "
    "Use the two available tools."
)

if HAS_OPENAI_KEY:
    messages = [{"role": "user", "content": user_prompt_ex2}]
    # First call — model may emit one or more function_call outputs.
    response = client.responses.create(model=MODEL_DEFAULT, input=messages, tools=tools_extended)
    # Process every function call, append the outputs, then re-invoke.
    name_to_fn = {"fetch_probabilities": fetch_probabilities, "compute_life_expectancy": compute_life_expectancy}
    any_calls = False
    for chunk in response.output:
        if hasattr(chunk, "name") and chunk.name in name_to_fn:
            any_calls = True
            args = json.loads(chunk.arguments)
            result = name_to_fn[chunk.name](**args)
            messages.append(chunk)
            messages.append({"type": "function_call_output", "call_id": chunk.call_id, "output": json.dumps(result)})
    if any_calls:
        final = client.responses.create(model=MODEL_DEFAULT, input=messages, tools=tools_extended)
        print(final.output_text)
    else:
        print(response.output_text)
else:
    print("[skipped — no OPENAI_API_KEY]")

### Exercise 3 — A fourth RAG query

Add one more query to the Generali pipeline: *"What is Generali's stated approach to climate-related underwriting risk?"* Define a Pydantic schema with two fields — `summary: str` and `key_initiatives: List[str]` — and run it through the same retrieve + augment + Structured-Outputs flow.

In [ ]:
# Solution

class ClimateApproach(BaseModel):
    summary: str
    key_initiatives: List[str]

query_climate = (
    "Extract Generali's stated approach to climate-related underwriting risk. "
    "Include any key initiatives, frameworks, or policies they explicitly mention."
)

if HAS_OPENAI_KEY and company_embeddings.get("Generali") is not None and not company_embeddings["Generali"].empty:
    climate_chunks = retrieve_top_chunks(company_embeddings["Generali"], query_climate)
    retrieved["climate"] = climate_chunks  # cache so query_with_rag picks it up
    result = query_with_rag("climate", query_climate, ClimateApproach)
    print(json.dumps(result.model_dump(), indent=2))
else:
    print("[skipped — no OPENAI_API_KEY or no embeddings index]")

## Summary

Mapped back to the four learning objectives:

- **Structured Outputs** — Pydantic schemas via `text_format=` give you machine-readable JSON without post-hoc parsing. Three stages (free-form → flat → nested) demonstrate the technique on a RISCBAC auto policy.
- **Function Calling** — expose Python functions as tools; the model decides when to call. Six-step round trip demonstrated on a mortality-table SQLite query.
- **Fine-Tuning** — adapts the model's weights to your domain. OpenAI wound down its self-serve platform in May 2026 — for new actuarial work, use **PEFT / LoRA on open-weights** (Llama 4, Mistral Small 4, Qwen 3.5, Gemma 4).
- **RAG** — load → clean → chunk → embed → retrieve → augment → respond. Demonstrated on a Generali annual-report PDF for solvency ratio, discount rates, and cyber-risk strategies. Combine with Structured Outputs to get typed, machine-readable answers.

In production, the four techniques compose. A claims-handling assistant might use RAG to ground in the policy wording, Function Calling to look up live claim status, Structured Outputs to return a typed decision record, and a fine-tuned model for the firm's tone of voice.

## Next steps

Continue with [Section 7 — Introduction to Agentic AI](../07_agentic_ai_introduction/07_agentic_ai_introduction.ipynb), which extends the building blocks here into agent loops: a model that plans, calls tools, observes results, replans, and proceeds — autonomously, across multiple steps.

## References

**Source notebook**
- *GenAI Beyond the Basics* (Deutsche Aktuarvereinigung), 2025 — [GitHub](https://github.com/DeutscheAktuarvereinigung/GenAI_Beyond_the_Basics). Sections 1–4 of this notebook are adapted from sections 3–6 there.

**Structured Outputs**
- Introducing Structured Outputs in the API — [OpenAI Blog (Aug 2024)](https://openai.com/index/introducing-structured-outputs-in-the-api/).
- Structured Outputs guide — [OpenAI docs](https://platform.openai.com/docs/guides/structured-outputs).
- Structured Outputs Cookbook — [openai/openai-cookbook](https://cookbook.openai.com/examples/structured_outputs_intro).

**Function Calling**
- Function Calling guide — [OpenAI docs](https://platform.openai.com/docs/guides/function-calling).
- How to Call Functions with Chat Models — [OpenAI Cookbook](https://cookbook.openai.com/examples/how_to_call_functions_with_chat_models).

**Fine-Tuning**
- *OpenAI is winding down the fine-tuning API and platform* — [community announcement, 2026-05-07](https://community.openai.com/t/openai-is-winding-down-the-fine-tuning-api-and-platform-discussion-thread/1380522).
- Hugging Face PEFT documentation — [huggingface.co/docs/peft](https://huggingface.co/docs/peft).
- Hugging Face fine-tuning cookbook — [huggingface.co/learn/cookbook](https://huggingface.co/learn/cookbook/en/fine_tuning_code_llm_on_single_gpu).

**Retrieval-Augmented Generation**
- *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* — Lewis et al., NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401).
- *From Local to Global: A Graph RAG Approach to Query-Focused Summarization* — Edge et al., Microsoft Research, 2024. [arXiv:2404.16130](https://arxiv.org/abs/2404.16130).
- LangChain RAG tutorial — [python.langchain.com](https://python.langchain.com/docs/tutorials/rag/).
- LlamaIndex documentation — [docs.llamaindex.ai](https://docs.llamaindex.ai/).

**Data sources used in this notebook**
- RISCBAC corpus (synthetic Québec auto-insurance contracts) — [Hugging Face](https://huggingface.co/datasets/davebulaval/RISCBAC).
- Generali Group Annual Integrated Report 2024 — [generali.com](https://www.generali.com).

All external URLs accessed 2026-05-26.

---

Part of the EAA seminar *Machine Learning & Generative AI: A Hands-On Guide to Actuarial Practice* by Dr. Simon Hatzesberger. Code under the MIT License — see [LICENSE](https://github.com/simonhatzesberger/ml-genai-actuarial-practice/blob/main/LICENSE).